In [0]:
import re
from collections import defaultdict
from datetime import datetime

raw_path = "s3://retail-sales-data-warehouse/raw/"

files = dbutils.fs.ls(raw_path)

grouped_files = defaultdict(list)

# STEP 1 — GROUP FILES

for file in files:

    file_name = file.name

    print("Checking:", file_name)

    # Example:
    # customers_src_07052026120000.csv

    match = re.match(
        r"(.+)_([0-9]{14})\.csv",
        file_name
    )

    if match:

        table_name = match.group(1)

        timestamp_str = match.group(2)

        # Convert DDMMYYYYHHMMSS → datetime
        timestamp = datetime.strptime(
            timestamp_str,
            "%d%m%Y%H%M%S"
        )

        grouped_files[table_name].append(
            (timestamp, file)
        )

# STEP 2 — KEEP LATEST FILE

for table_name, file_list in grouped_files.items():

    print(f"\nProcessing: {table_name}")

    # Sort newest first
    sorted_files = sorted(
        file_list,
        key=lambda x: x[0],
        reverse=True
    )

    latest_file = sorted_files[0][1]

    print("Keeping Latest:", latest_file.name)

    # Archive older files
    for _, old_file in sorted_files[1:]:

        source_path = old_file.path

        archive_path = (
            f"s3://retail-sales-data-warehouse/archive/"
            f"{table_name}/"
            f"{old_file.name}"
        )

        print("Archiving:", old_file.name)

        dbutils.fs.mv(
            source_path,
            archive_path
        )

print("\nIncremental archival completed successfully")